In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# path constants
DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

IBI_MIN = 300
IBI_MAX = 2000
RMSSD_THRESHOLD = 20   # ms, Task Force (1996) low parasympathetic tone
SDNN_THRESHOLD = 50    # ms, Task Force (1996) reduced autonomic variability

def calculate_hrv_metrics(ibi_data):
    s = pd.Series(ibi_data)
    rmssd = np.sqrt(np.mean(np.square(np.diff(s))))
    sdnn = np.std(s, ddof=1)
    return rmssd, sdnn

def get_ibi_data(questions, ibi_data):
    ibi = ibi_data.copy()
    ibi['datetime'] = pd.to_datetime(ibi['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    mask = (
        (ibi['datetime'] >= questions['Question Start Time'].min()) &
        (ibi['datetime'] <= questions['Question Answer Time'].max())
    )
    vals = ibi.loc[mask, 'ibi'].values
    return vals[(vals > IBI_MIN) & (vals < IBI_MAX)]

def determine_anxiety(rmssd, sdnn, rmssd_baseline, sdnn_baseline,
                      rmssd_threshold=RMSSD_THRESHOLD, sdnn_threshold=SDNN_THRESHOLD):
    general = rmssd < rmssd_threshold or sdnn < sdnn_threshold
    individual = rmssd < rmssd_baseline or sdnn < sdnn_baseline
    return general, individual

def process_questions(question_type, psychometric_data, ibi_data, rmssd_baseline, sdnn_baseline):
    yn = lambda x: 'Yes' if x else 'No'
    rows = []
    for i, (psych, ibi) in enumerate(zip(psychometric_data, ibi_data), 1):
        qs = psych[psych['Type'] == question_type].copy()
        ibi_q = get_ibi_data(qs, ibi)
        rmssd, sdnn = calculate_hrv_metrics(ibi_q)
        rows.append({
            'Test': f'Test {i:02d}',
            'Start Time': qs['Question Start Time'].min().strftime('%H:%M:%S'),
            'End Time': qs['Question Answer Time'].max().strftime('%H:%M:%S'),
            'RMSSD': round(rmssd, 2),
            'SDNN': round(sdnn, 2),
            'General Anxiety (RMSSD)': yn(rmssd < RMSSD_THRESHOLD),
            'General Anxiety (SDNN)': yn(sdnn < SDNN_THRESHOLD),
            'Individual Anxiety (RMSSD)': yn(rmssd < rmssd_baseline),
            'Individual Anxiety (SDNN)': yn(sdnn < sdnn_baseline),
        })
    return pd.DataFrame(rows)

def load_psychometric(path):
    df = pd.read_csv(path)
    for col in ['Question Start Time', 'Question Answer Time']:
        df[col] = pd.to_datetime(df[col], utc=True, errors='coerce').dt.tz_convert(None)
    return df.dropna(subset=['Question Start Time'])

# load IBI
ibi_baseline = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

# validate baseline IBI
ibi_baseline = ibi_baseline[(ibi_baseline['ibi'] > IBI_MIN) & (ibi_baseline['ibi'] < IBI_MAX)]

# baseline HRV
rmssd_baseline, sdnn_baseline = calculate_hrv_metrics(ibi_baseline['ibi'])

psychometric_01 = load_psychometric(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = load_psychometric(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = load_psychometric(f'{PSY}/Psychometric_Test_Results_03.csv')

psychometric_data = [psychometric_01, psychometric_02, psychometric_03]
ibi_data = [ibi_01, ibi_02, ibi_03]
question_types = ['HADS', 'STAI-S', 'STAI-T', 'BFI', 'FQ']

print('Setup complete.')

In [ ]:
# per-session HRV
def _filter(df):
    return df[(df['ibi'] > IBI_MIN) & (df['ibi'] < IBI_MAX)]['ibi']

rmssd_01, sdnn_01 = calculate_hrv_metrics(_filter(ibi_01))
rmssd_02, sdnn_02 = calculate_hrv_metrics(_filter(ibi_02))
rmssd_03, sdnn_03 = calculate_hrv_metrics(_filter(ibi_03))

gen_01, _ = determine_anxiety(rmssd_01, sdnn_01, rmssd_baseline, sdnn_baseline)
gen_02, _ = determine_anxiety(rmssd_02, sdnn_02, rmssd_baseline, sdnn_baseline)
gen_03, _ = determine_anxiety(rmssd_03, sdnn_03, rmssd_baseline, sdnn_baseline)

sessions = {
    'Baseline': (rmssd_baseline, sdnn_baseline, False),
    'Test 01': (rmssd_01, sdnn_01, gen_01),
    'Test 02': (rmssd_02, sdnn_02, gen_02),
    'Test 03': (rmssd_03, sdnn_03, gen_03),
}

print(f'Baseline  RMSSD: {rmssd_baseline:.2f} ms  SDNN: {sdnn_baseline:.2f} ms')
for label, (r, s, g) in list(sessions.items())[1:]:
    print(f'{label}  RMSSD: {r:.2f} ms  SDNN: {s:.2f} ms  Anxiety: {"Yes" if g else "No"}')

labels = list(sessions.keys())
colors = ['red' if v[2] else 'green' for v in sessions.values()]

fig, axes = plt.subplots(2, 1, figsize=(10, 8))

axes[0].bar(labels, [v[0] for v in sessions.values()], color=colors, alpha=0.7)
axes[0].axhline(RMSSD_THRESHOLD, color='red', linestyle='--', label='Threshold')
axes[0].set_title('RMSSD Across Sessions')
axes[0].set_ylabel('RMSSD (ms)')
axes[0].legend()

axes[1].bar(labels, [v[1] for v in sessions.values()], color=colors, alpha=0.7)
axes[1].axhline(SDNN_THRESHOLD, color='red', linestyle='--', label='Threshold')
axes[1].set_title('SDNN Across Sessions')
axes[1].set_ylabel('SDNN (ms)')
axes[1].legend()

plt.tight_layout()
plt.show()
plt.close()

In [ ]:
# all question types
all_results = []
for qt in question_types:
    df = process_questions(qt, psychometric_data, ibi_data, rmssd_baseline, sdnn_baseline)
    df.insert(1, 'Type', qt)
    all_results.append(df)

all_results = pd.concat(all_results, ignore_index=True)
print(f'Baseline RMSSD: {rmssd_baseline:.2f} ms, SDNN: {sdnn_baseline:.2f} ms\n')
print(all_results.to_string(index=False))